# 01 · Exploración — `data/renta_ine`

**Pipeline:** `01_exploracion` → `02_limpieza` → `03_visualizacion`  
**Objetivo:** entender estructura, encoding, separadores, tipos de columna,
NaN patterns y cardinalidades de los 6 ficheros del INE antes de limpiar.

| Fichero | Alias |
|---------|-------|
| `1.Indicadores de renta media y mediana.csv` | `renta` |
| `2.Distribución por fuente de ingresos.csv` | `fuente_ing` |
| `4.Porcentaje de población ... edad.csv` | `umbrales_edad` |
| `5.Porcentaje de población ... nacionalidad.csv` | `umbrales_nacionalid` |
| `9.Índice de Gini y Distribución de la renta P80P20.csv` | `gini` |
| `10.Indicadores demográficos.csv` | `demografico` |


In [1]:
#=====================================================================
# CELDA 1: Carga PATH
#=====================================================================


# ── Librerías ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# ── Estilo visual ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# ── Rutas (auto-discovery: sube hasta encontrar /data) ────────────────────────
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR   = PROJECT_ROOT / 'data' / 'renta_ine'
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'clean'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATA_DIR     :', DATA_DIR)
print('OUTPUT_DIR   :', OUTPUT_DIR)
print('DATA existe  :', DATA_DIR.exists())

# ── Ficheros ──────────────────────────────────────────────────────────────────
FILES = {
    'renta':               DATA_DIR / '1.Indicadores de renta media y mediana.csv',
    'fuente_ing':          DATA_DIR / '2.Distribución por fuente de ingresos.csv',
    'umbrales_edad':       DATA_DIR / '4.Porcentaje de población con ingresos por unidad de consumo por debajo de determinados umbrales fijos por sexo y tramos de edad.csv',
    'umbrales_nacionalid': DATA_DIR / '5.Porcentaje de población con ingresos por unidad de consumo por debajo de determinados umbrales fijos por sexo y nacionalidad.csv',
    'gini':                DATA_DIR / '9.Índice de Gini y Distribución de la renta P80P20.csv',
    'demografico':         DATA_DIR / '10.Indicadores demográficos.csv',
}

# ── Constantes ────────────────────────────────────────────────────────────────
ENCODING  = 'utf-8-sig'
SEPARATOR = ';'
NA_VALUES = ['..', '']

# ── Mapa código INE → nombre provincia ───────────────────────────────────────
COD_PROVINCIA = {
    '01':'Álava',        '02':'Albacete',     '03':'Alicante',    '04':'Almería',
    '05':'Ávila',        '06':'Badajoz',      '07':'Baleares',    '08':'Barcelona',
    '09':'Burgos',       '10':'Cáceres',      '11':'Cádiz',       '12':'Castellón',
    '13':'Ciudad Real',  '14':'Córdoba',      '15':'A Coruña',    '16':'Cuenca',
    '17':'Girona',       '18':'Granada',      '19':'Guadalajara', '20':'Gipuzkoa',
    '21':'Huelva',       '22':'Huesca',       '23':'Jaén',        '24':'León',
    '25':'Lleida',       '26':'La Rioja',     '27':'Lugo',        '28':'Madrid',
    '29':'Málaga',       '30':'Murcia',       '31':'Navarra',     '32':'Ourense',
    '33':'Asturias',     '34':'Palencia',     '35':'Las Palmas',  '36':'Pontevedra',
    '37':'Salamanca',    '38':'S.C. Tenerife','39':'Cantabria',   '40':'Segovia',
    '41':'Sevilla',      '42':'Soria',        '43':'Tarragona',   '44':'Teruel',
    '45':'Toledo',       '46':'Valencia',     '47':'Valladolid',  '48':'Bizkaia',
    '49':'Zamora',       '50':'Zaragoza',     '51':'Ceuta',       '52':'Melilla',
}

# ── Verificación final ────────────────────────────────────────────────────────
print('\n🔍 Verificando ficheros:')
all_ok = True
for nombre, path in FILES.items():
    estado = '✅' if path.exists() else '❌ NO ENCONTRADO'
    if not path.exists(): all_ok = False
    print(f'  {estado}  {nombre:<20} → {path.name[:60]}')

print('\n✅ Configuración lista — todos los ficheros encontrados' if all_ok else
      '\n❌ Revisa los ficheros marcados arriba antes de continuar')

PROJECT_ROOT : c:\Users\Daniel Bolaños\OneDrive\Documentos\GitHub\TFG_Spain-s_Migratory_Flow
DATA_DIR     : c:\Users\Daniel Bolaños\OneDrive\Documentos\GitHub\TFG_Spain-s_Migratory_Flow\data\renta_ine
OUTPUT_DIR   : c:\Users\Daniel Bolaños\OneDrive\Documentos\GitHub\TFG_Spain-s_Migratory_Flow\output\clean
DATA existe  : True

🔍 Verificando ficheros:
  ✅  renta                → 1.Indicadores de renta media y mediana.csv
  ✅  fuente_ing           → 2.Distribución por fuente de ingresos.csv
  ✅  umbrales_edad        → 4.Porcentaje de población con ingresos por unidad de consumo
  ✅  umbrales_nacionalid  → 5.Porcentaje de población con ingresos por unidad de consumo
  ✅  gini                 → 9.Índice de Gini y Distribución de la renta P80P20.csv
  ✅  demografico          → 10.Indicadores demográficos.csv

✅ Configuración lista — todos los ficheros encontrados


In [2]:
#=====================================================================
# CELDA 2: Carga cruda de los 6 CSV y metadata inicial
# (shape, dtypes, nulos, muestra) para orientar la limpieza
#=====================================================================

def metadata_csv(nombre, path):
    df = pd.read_csv(path, sep=SEPARATOR, encoding=ENCODING, na_values=NA_VALUES)
    total_celdas = df.shape[0] * df.shape[1]
    nulos_total  = df.isnull().sum().sum()
    print(f"\n{'='*65}")
    print(f"📄 {nombre.upper()}  →  {path.name}")
    print(f"{'='*65}")
    print(f"  Shape        : {df.shape[0]} filas × {df.shape[1]} columnas")
    print(f"  Nulos totales: {nulos_total}  ({nulos_total/total_celdas*100:.1f}% del total)")
    print(f"\n  — Nulos por columna —")
    nulos_col = df.isnull().sum()
    nulos_col = nulos_col[nulos_col > 0]
    if nulos_col.empty:
        print("    (ninguna columna con nulos)")
    else:
        for col, n in nulos_col.items():
            print(f"    {col:<50} {n:>6}  ({n/df.shape[0]*100:.1f}%)")
    print(f"\n  — Tipos de dato —")
    for col, dtype in df.dtypes.items():
        print(f"    {col:<50} {str(dtype):<10}")
    print(f"\n  — Primeras 3 filas —")
    print(df.head(3).to_string(index=False))
    return df

# Cargamos y guardamos cada DF en un diccionario para usarlo en celdas siguientes
RAW = {}
for nombre, path in FILES.items():
    RAW[nombre] = metadata_csv(nombre, path)

print(f"\n\n✅ Carga completa — {len(RAW)} DataFrames disponibles en RAW[...]")


📄 RENTA  →  1.Indicadores de renta media y mediana.csv
  Shape        : 3009312 filas × 6 columnas
  Nulos totales: 1532358  (8.5% del total)

  — Nulos por columna —
    Distritos                                          439506  (14.6%)
    Secciones                                          1007424  (33.5%)
    Total                                               85428  (2.8%)

  — Tipos de dato —
    Municipios                                         str       
    Distritos                                          str       
    Secciones                                          str       
    Indicadores de renta media                         str       
    Periodo                                            int64     
    Total                                              str       

  — Primeras 3 filas —
            Municipios Distritos Secciones   Indicadores de renta media  Periodo  Total
01001 Alegría-Dulantzi       NaN       NaN Renta neta media por persona     2023 16.429
01

In [3]:
#=====================================================================
# CELDA 3: Diagnóstico profundo — valores únicos de Total (tokens NaN
# no capturados) y desglose de niveles geográficos por CSV
#=====================================================================

print("=" * 65)
print("🔎 TOKENS RAROS EN COLUMNA 'Total' (valores no numéricos)")
print("=" * 65)

for nombre, df in RAW.items():
    # Intentamos convertir Total a float para ver qué falla
    total_str = df['Total'].astype(str).str.replace(',', '.', regex=False)
    mask_fallo = pd.to_numeric(total_str, errors='coerce').isna() & df['Total'].notna()
    tokens = df.loc[mask_fallo, 'Total'].value_counts()
    print(f"\n  📄 {nombre:<22} → {mask_fallo.sum()} celdas no convertibles")
    if not tokens.empty:
        print(tokens.to_string())

print("\n" + "=" * 65)
print("🗺️  DESGLOSE GEOGRÁFICO — cuántas filas hay por nivel")
print("=" * 65)

for nombre, df in RAW.items():
    solo_mun      = df['Distritos'].isna() & df['Secciones'].isna()
    solo_dist     = df['Distritos'].notna() & df['Secciones'].isna()
    hasta_seccion = df['Secciones'].notna()
    print(f"\n  📄 {nombre}")
    print(f"    Nivel municipio  : {solo_mun.sum():>10,} filas")
    print(f"    Nivel distrito   : {solo_dist.sum():>10,} filas")
    print(f"    Nivel sección    : {hasta_seccion.sum():>10,} filas")
    print(f"    TOTAL            : {len(df):>10,} filas")

print("\n" + "=" * 65)
print("📅 RANGO DE AÑOS DISPONIBLES POR CSV")
print("=" * 65)
for nombre, df in RAW.items():
    años = sorted(df['Periodo'].dropna().unique())
    print(f"  {nombre:<22} → {años[0]}–{años[-1]}  ({len(años)} años: {años})")

🔎 TOKENS RAROS EN COLUMNA 'Total' (valores no numéricos)

  📄 renta                  → 162132 celdas no convertibles
Total
.    162132

  📄 fuente_ing             → 200690 celdas no convertibles
Total
.    200690

  📄 umbrales_edad          → 11596950 celdas no convertibles
Total
.    11596950

  📄 umbrales_nacionalid    → 8379249 celdas no convertibles
Total
.    8379249

  📄 gini                   → 73532 celdas no convertibles
Total
.    73532

  📄 demografico            → 143493 celdas no convertibles
Total
.            143475
1.613.579         1
1.597.238         1
1.576.831         1
1.588.157         1
1.606.253         1
1.595.747         1
1.584.505         1
1.575.453         1
1.581.200         1
3.362.335         1
3.293.178         1
3.230.315         1
3.240.998         1
3.277.392         1
3.218.339         1
3.173.391         1
3.131.879         1
3.121.185         1

🗺️  DESGLOSE GEOGRÁFICO — cuántas filas hay por nivel

  📄 renta
    Nivel municipio  :    439,506 fil

In [4]:
#=====================================================================
# CELDA 4: Inspección de filas con valores raros en Total —
# ver el contexto completo para decidir cómo tratarlos
#=====================================================================

print("=" * 75)
print("🔎 CASO A — El punto '.' en todos los CSVs")
print("  Hipótesis: dato no disponible para ese municipio/distrito/sección")
print("=" * 75)

# Tomamos 'renta' como representativo, mostramos 10 ejemplos variados
df_r = RAW['renta']
mask_punto = df_r['Total'].astype(str).str.strip() == '.'
muestra_punto = df_r[mask_punto].sample(10, random_state=42)
print("\n  10 filas aleatorias con Total == '.' en RENTA:")
print(muestra_punto.to_string(index=False))

# ¿El '.' aparece en municipios, distritos o secciones específicas?
print("\n  ¿A qué nivel geográfico aparece el '.' en RENTA?")
print(df_r[mask_punto][['Distritos','Secciones']].isna().sum())

print("\n" + "=" * 75)
print("🔎 CASO B — Números con puntos de miles en DEMOGRAFICO")
print("  Hipótesis: formato 1.613.579 = número grande con separador de miles")
print("=" * 75)

df_d = RAW['demografico']
total_str = df_d['Total'].astype(str).str.strip()
# Detectamos el patrón X.XXX.XXX o X.XXX
mask_miles = total_str.str.match(r'^\d{1,3}(\.\d{3})+$')
muestra_miles = df_d[mask_miles][['Municipios','Distritos','Secciones',
                                   'Indicadores demográficos','Periodo','Total']].head(20)
print(f"\n  Filas con formato 'miles' detectadas: {mask_miles.sum()}")
print(muestra_miles.to_string(index=False))

print("\n  ¿Qué indicadores demográficos tienen este formato?")
print(df_d[mask_miles]['Indicadores demográficos'].value_counts().to_string())

print("\n  ¿Y los mismos indicadores con valor normal (sin puntos de miles)?")
indicadores_miles = df_d[mask_miles]['Indicadores demográficos'].unique()
for ind in indicadores_miles:
    muestra_normal = df_d[
        (df_d['Indicadores demográficos'] == ind) & ~mask_miles & df_d['Total'].notna()
    ]['Total'].head(5).tolist()
    print(f"\n    '{ind}'  →  ejemplos sin formato miles: {muestra_normal}")
    

🔎 CASO A — El punto '.' en todos los CSVs
  Hipótesis: dato no disponible para ese municipio/distrito/sección

  10 filas aleatorias con Total == '.' en RENTA:
             Municipios                            Distritos                                 Secciones                Indicadores de renta media  Periodo Total
     09445 Villambistia     0944501 Villambistia distrito 01     0944501001 Villambistia sección 01001               Renta bruta media por hogar     2019     .
       42176 Tajahuerce                                  NaN                                       NaN Mediana de la renta por unidad de consumo     2015     .
       37119 Encina, La                                  NaN                                       NaN   Media de la renta por unidad de consumo     2018     .
47170 Torre de Peñafiel                                  NaN                                       NaN   Media de la renta por unidad de consumo     2018     .
             31175 Mues                 

In [5]:
#=====================================================================
# CELDA 5: % de NaN reales por columna y por CSV — separando
# NaN estructurales (niveles geo opcionales) de NaN en Total
# y distinguiendo '.' (secreto estadístico) de NaN puro
#=====================================================================

print("=" * 75)
print("📊 ANÁLISIS COMPLETO DE NULOS POR CSV")
print("=" * 75)

for nombre, df in RAW.items():
    total_filas = len(df)
    print(f"\n{'─'*75}")
    print(f"  📄 {nombre.upper()}  —  {total_filas:,} filas")
    print(f"{'─'*75}")

    for col in df.columns:
        nulos       = df[col].isna().sum()
        pct_nulo    = nulos / total_filas * 100

        if col == 'Total':
            # Separamos: NaN puro vs secreto estadístico '.'
            secreto     = (df[col].astype(str).str.strip() == '.').sum()
            nan_puro    = nulos  # los que ya vienen como NaN (estaban en NA_VALUES)
            convertible = total_filas - nulos - secreto
            print(f"    {'Total':<52} NaN puro: {nan_puro:>7,} ({nan_puro/total_filas*100:>5.1f}%)  |  "
                  f"Secreto '.': {secreto:>7,} ({secreto/total_filas*100:>5.1f}%)  |  "
                  f"Convertible: {convertible:>7,} ({convertible/total_filas*100:>5.1f}%)")
        elif col in ('Distritos', 'Secciones'):
            # NaN estructural — son niveles geo no siempre presentes
            print(f"    {col:<52} NaN estructural: {nulos:>7,} ({pct_nulo:>5.1f}%)  ← nivel geo opcional")
        else:
            print(f"    {col:<52} NaN: {nulos:>7,} ({pct_nulo:>5.1f}%)")

print(f"\n{'='*75}")
print("📝 RESUMEN EJECUTIVO")
print(f"{'='*75}")
resumen = []
for nombre, df in RAW.items():
    total_filas = len(df)
    secreto_total = (df['Total'].astype(str).str.strip() == '.').sum()
    nan_total     = df['Total'].isna().sum()
    convertible   = total_filas - nan_total - secreto_total
    resumen.append({
        'CSV'             : nombre,
        'Filas'           : total_filas,
        'Total convertible (%)' : f"{convertible/total_filas*100:.1f}%",
        'Secreto estadístico (%)': f"{secreto_total/total_filas*100:.1f}%",
        'NaN puro Total (%)': f"{nan_total/total_filas*100:.1f}%",
    })

df_resumen = pd.DataFrame(resumen).set_index('CSV')
print(df_resumen.to_string())

📊 ANÁLISIS COMPLETO DE NULOS POR CSV

───────────────────────────────────────────────────────────────────────────
  📄 RENTA  —  3,009,312 filas
───────────────────────────────────────────────────────────────────────────
    Municipios                                           NaN:       0 (  0.0%)
    Distritos                                            NaN estructural: 439,506 ( 14.6%)  ← nivel geo opcional
    Secciones                                            NaN estructural: 1,007,424 ( 33.5%)  ← nivel geo opcional
    Indicadores de renta media                           NaN:       0 (  0.0%)
    Periodo                                              NaN:       0 (  0.0%)
    Total                                                NaN puro:  85,428 (  2.8%)  |  Secreto '.': 162,132 (  5.4%)  |  Convertible: 2,761,752 ( 91.8%)

───────────────────────────────────────────────────────────────────────────
  📄 FUENTE_ING  —  2,507,850 filas
─────────────────────────────────────────────────

In [6]:
#=====================================================================
# CELDA 6: Exploración final antes de limpieza —
# cobertura geográfica, categorías, consistencia temporal
# y relación secreto estadístico vs tamaño de municipio
#=====================================================================

# ── Utilidad: extraer código de municipio (primeros 5 chars) ─────────
def cod_mun(serie):
    return serie.astype(str).str.strip().str[:5]

# ─────────────────────────────────────────────────────────────────────
# 1. COBERTURA GEOGRÁFICA — municipios únicos por CSV
# ─────────────────────────────────────────────────────────────────────
print("=" * 75)
print("1️⃣  COBERTURA GEOGRÁFICA — municipios únicos por CSV")
print("=" * 75)

sets_municipios = {}
for nombre, df in RAW.items():
    muns = set(cod_mun(df['Municipios']))
    sets_municipios[nombre] = muns
    print(f"  {nombre:<22} → {len(muns):>5} municipios únicos")

# Municipios presentes en TODOS los CSV
interseccion = set.intersection(*sets_municipios.values())
union        = set.union(*sets_municipios.values())
print(f"\n  ✅ En todos los CSV (intersección) : {len(interseccion):>5} municipios")
print(f"  🔵 En algún CSV  (unión)           : {len(union):>5} municipios")
print(f"  📌 Referencia INE España           :  8.131 municipios")

# ¿Qué CSV tiene municipios que otros no tienen?
print("\n  — Municipios exclusivos por CSV (no están en los demás) —")
for nombre, s in sets_municipios.items():
    exclusivos = s - set.union(*[v for k,v in sets_municipios.items() if k != nombre])
    print(f"  {nombre:<22} → {len(exclusivos):>4} exclusivos")

# ─────────────────────────────────────────────────────────────────────
# 2. VALORES ÚNICOS DE COLUMNAS CATEGÓRICAS
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print("2️⃣  CATEGORÍAS — indicadores y dimensiones por CSV")
print("=" * 75)

CATS = {
    'renta'              : ['Indicadores de renta media'],
    'fuente_ing'         : ['Distribución por fuente de ingresos'],
    'umbrales_edad'      : ['Sexo', 'Tramos de edad',
                            'Distribución de la renta por unidad de consumo'],
    'umbrales_nacionalid': ['Sexo', 'Nacionalidad',
                            'Distribución de la renta por unidad de consumo'],
    'gini'               : ['Indicadores de renta media'],
    'demografico'        : ['Indicadores demográficos'],
}

for nombre, cols in CATS.items():
    print(f"\n  📄 {nombre.upper()}")
    for col in cols:
        vals = RAW[nombre][col].dropna().unique()
        print(f"    [{col}]  —  {len(vals)} valores únicos:")
        for v in sorted(vals):
            print(f"      · {v}")

# ─────────────────────────────────────────────────────────────────────
# 3. CONSISTENCIA TEMPORAL — ¿todos los municipios tienen 9 años?
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print("3️⃣  CONSISTENCIA TEMPORAL — huecos por municipio")
print("=" * 75)

AÑOS_ESPERADOS = 9

for nombre, df in RAW.items():
    # Nivel municipio únicamente para no mezclar con distritos/secciones
    df_mun = df[df['Distritos'].isna() & df['Secciones'].isna()].copy()
    df_mun['cod_mun'] = cod_mun(df_mun['Municipios'])

    # Primer indicador/categoría del CSV para contar años sin duplicar
    primera_col_cat = list(CATS[nombre])[0]
    primer_valor    = df_mun[primera_col_cat].dropna().unique()[0]
    df_filtrado     = df_mun[df_mun[primera_col_cat] == primer_valor]

    años_por_mun    = df_filtrado.groupby('cod_mun')['Periodo'].nunique()
    completos       = (años_por_mun == AÑOS_ESPERADOS).sum()
    incompletos     = (años_por_mun  < AÑOS_ESPERADOS).sum()

    print(f"\n  📄 {nombre:<22}  (indicador: '{primer_valor[:50]}')")
    print(f"    Municipios con 9 años completos : {completos:>5}")
    print(f"    Municipios con años incompletos : {incompletos:>5}")
    if incompletos > 0:
        detalle = años_por_mun[años_por_mun < AÑOS_ESPERADOS].value_counts().sort_index()
        print(f"    Distribución de años disponibles:")
        for n_años, count in detalle.items():
            print(f"      {n_años} años → {count:>4} municipios")

# ─────────────────────────────────────────────────────────────────────
# 4. SECRETO ESTADÍSTICO vs TAMAÑO DE MUNICIPIO
#    Usamos 'demografico' indicador Población como proxy de tamaño
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print("4️⃣  SECRETO ESTADÍSTICO vs TAMAÑO DE MUNICIPIO")
print("   (proxy: Población 2023 del CSV demográfico, nivel municipio)")
print("=" * 75)

df_d = RAW['demografico'].copy()
df_d['cod_mun'] = cod_mun(df_d['Municipios'])

# Población 2023 nivel municipio — quitamos puntos de miles
df_pob = df_d[
    (df_d['Indicadores demográficos'] == 'Población') &
    (df_d['Periodo'] == 2023) &
    df_d['Distritos'].isna() &
    df_d['Secciones'].isna()
].copy()
df_pob['poblacion'] = (
    df_pob['Total'].astype(str).str.replace('.', '', regex=False)
                   .str.replace(',', '.', regex=False)
)
df_pob['poblacion'] = pd.to_numeric(df_pob['poblacion'], errors='coerce')

# Para cada CSV calculamos % secreto por tramo de tamaño
TRAMOS = [0, 500, 1000, 5000, 10000, 50000, float('inf')]
ETIQ   = ['<500', '500-1k', '1k-5k', '5k-10k', '10k-50k', '>50k']

for nombre, df in RAW.items():
    df_mun = df[df['Distritos'].isna() & df['Secciones'].isna()].copy()
    df_mun['cod_mun'] = cod_mun(df_mun['Municipios'])
    df_mun['es_secreto'] = df_mun['Total'].astype(str).str.strip() == '.'

    # Merge con población
    merged = df_mun.merge(df_pob[['cod_mun','poblacion']], on='cod_mun', how='left')
    merged['tramo'] = pd.cut(merged['poblacion'], bins=TRAMOS, labels=ETIQ)

    resumen = merged.groupby('tramo', observed=True)['es_secreto'].agg(
        total='count', secretos='sum'
    )
    resumen['pct_secreto'] = (resumen['secretos'] / resumen['total'] * 100).round(1)

    print(f"\n  📄 {nombre.upper()}")
    print(resumen[['total','secretos','pct_secreto']].to_string())

print("\n\n✅ Exploración completa — listos para diseñar la limpieza")

1️⃣  COBERTURA GEOGRÁFICA — municipios únicos por CSV
  renta                  →  8139 municipios únicos
  fuente_ing             →  8139 municipios únicos
  umbrales_edad          →  8139 municipios únicos
  umbrales_nacionalid    →  8139 municipios únicos
  gini                   →  8139 municipios únicos
  demografico            →  8139 municipios únicos

  ✅ En todos los CSV (intersección) :  8139 municipios
  🔵 En algún CSV  (unión)           :  8139 municipios
  📌 Referencia INE España           :  8.131 municipios

  — Municipios exclusivos por CSV (no están en los demás) —
  renta                  →    0 exclusivos
  fuente_ing             →    0 exclusivos
  umbrales_edad          →    0 exclusivos
  umbrales_nacionalid    →    0 exclusivos
  gini                   →    0 exclusivos
  demografico            →    0 exclusivos

2️⃣  CATEGORÍAS — indicadores y dimensiones por CSV

  📄 RENTA
    [Indicadores de renta media]  —  6 valores únicos:
      · Media de la renta por unida